# Preparación del RAG — Normativa de IA en España

Pipeline para preparar la base documental de un RAG:

1. Carga de documentos desde `data/`
2. Limpieza del texto
3. Troceado (chunking) con solapamiento
4. Embeddings con un modelo de HuggingFace ejecutado en local
5. Construcción y persistencia de un índice FAISS
6. Prueba de recuperación

## Antes de empezar

Coloca los documentos en la carpeta `data/`. Formatos soportados: **PDF, TXT, MD, HTML**.

## 1. Dependencias

In [ ]:
%pip install -q pypdf sentence-transformers faiss-cpu numpy

## 2. Imports y configuración

In [ ]:
import json
import re
from pathlib import Path

import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("data")
OUTPUT_DIR = Path("rag_store")
OUTPUT_DIR.mkdir(exist_ok=True)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150
SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md", ".html", ".htm"}

## 3. Carga de documentos

In [ ]:
def read_pdf(path):
    reader = PdfReader(str(path))
    return "\n".join((page.extract_text() or "") for page in reader.pages)


def read_html(path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    text = re.sub(r"(?is)<(script|style).*?>.*?</\1>", " ", text)
    text = re.sub(r"(?s)<[^>]+>", " ", text)
    return text


def read_text(path):
    return path.read_text(encoding="utf-8", errors="ignore")


def load_documents(data_dir):
    if not data_dir.exists():
        raise FileNotFoundError(
            f"No existe la carpeta '{data_dir}'. Crea 'data/' y añade PDFs, TXT, MD o HTML."
        )

    documents = []
    for path in sorted(data_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        if path.suffix.lower() == ".pdf":
            content = read_pdf(path)
        elif path.suffix.lower() in {".html", ".htm"}:
            content = read_html(path)
        else:
            content = read_text(path)

        documents.append({"source": str(path), "content": content})

    return documents


documents = load_documents(DATA_DIR)
print(f"Documentos cargados: {len(documents)}")
for doc in documents:
    print(f"- {doc['source']} ({len(doc['content'])} caracteres)")

## 4. Limpieza del texto

Se eliminan guiones de corte, espacios repetidos y saltos de línea excesivos.

In [ ]:
def clean_text(text):
    text = text.replace("\u00ad", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

## 5. Chunking

Se divide por frases respetando el tamaño máximo y añadiendo un solapamiento entre chunks para no perder contexto en los cortes.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, current = [], ""

    for sentence in sentences:
        if not sentence:
            continue

        if len(sentence) > chunk_size:
            step = max(chunk_size - overlap, 1)
            for i in range(0, len(sentence), step):
                chunks.append(sentence[i:i + chunk_size])
            current = ""
            continue

        if not current:
            current = sentence
        elif len(current) + 1 + len(sentence) <= chunk_size:
            current = f"{current} {sentence}"
        else:
            chunks.append(current)
            tail = current[-overlap:] if overlap else ""
            current = f"{tail} {sentence}".strip()

    if current:
        chunks.append(current)

    return chunks


chunks = []
for doc in documents:
    cleaned = clean_text(doc["content"])
    for chunk_id, chunk in enumerate(chunk_text(cleaned)):
        chunks.append(
            {
                "source": doc["source"],
                "chunk_id": chunk_id,
                "text": chunk,
            }
        )

print(f"Chunks generados: {len(chunks)}")
if not chunks:
    raise ValueError(
        "No se han generado chunks. Revisa que los documentos tengan texto extraíble."
    )

## 6. Embeddings en local

Se usa `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, un modelo multilingüe que funciona bien para textos en español y no requiere API.

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)

texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")

print(f"Dimensiones de embeddings: {embeddings.shape}")

## 7. Construcción del índice FAISS

Se usa `IndexFlatIP` (producto interno) junto con embeddings normalizados, lo que equivale a similitud coseno.

In [ ]:
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

index_path = OUTPUT_DIR / "index.faiss"
chunks_path = OUTPUT_DIR / "chunks.json"
meta_path = OUTPUT_DIR / "metadata.json"

faiss.write_index(index, str(index_path))
chunks_path.write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8"
)
meta_path.write_text(
    json.dumps(
        {
            "embedding_model": EMBEDDING_MODEL,
            "num_documents": len(documents),
            "num_chunks": len(chunks),
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "dimension": int(embeddings.shape[1]),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(f"Índice guardado en: {index_path}")
print(f"Chunks guardados en: {chunks_path}")
print(f"Metadatos guardados en: {meta_path}")

## 8. Prueba de recuperación

Comprueba que el índice devuelve fragmentos relevantes para una consulta de ejemplo.

In [ ]:
def search(query, k=5):
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    scores, indices = index.search(query_embedding, k)
    return [
        (float(score), chunks[index])
        for score, index in zip(scores[0], indices[0])
        if index != -1
    ]


for score, chunk in search("¿Qué obligaciones establece la normativa de IA en España?", k=5):
    print(f"[{score:.3f}] {chunk['source']} (chunk {chunk['chunk_id']})")
    print(chunk["text"][:400])
    print("-" * 80)

## 9. Cómo reutilizar el índice

El índice queda guardado en `rag_store/`. Para usarlo en otro notebook o script:

```python
import json
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
index = faiss.read_index("rag_store/index.faiss")
chunks = json.loads(open("rag_store/chunks.json", encoding="utf-8").read())

def search(query, k=5):
    emb = model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(emb, k)
    return [(float(s), chunks[i]) for s, i in zip(scores[0], ids[0]) if i != -1]
```

Con los chunks recuperados ya puedes construir el prompt y pasárselo a tu LLM.